# WithGyro Experiment 0.1 — Paired channel-ablation temporal representation probe

## Research question

How much discriminative information is carried by linear-acceleration events, gyro-derived angular-acceleration events, and their combination across different temporal aggregation scales?

Paired channel sets:

- `accel30`: channels `0:30`;
- `angular30`: channels `30:60`;
- `combined60`: channels `0:60`;
- raw IMU channels `60:66` are excluded.

The primary gyro-contribution contrast is `combined60 - accel30`, computed within each split before aggregation.

## Protocol

- dataset: Angular66 action-0 output at 64 Hz;
- split seeds: `(11, 23, 37, 53, 71)` with 12/4/4 train/validation/test users;
- fixed-duration sweep: `50, 150, 250, 350, 450, 550, 650, 750, 850, 950, 1050 ms`;
- relative-progress sweep: `1, 2, 4, 6, 8, 10, 12, 16, 20` bins;
- feature: channel-wise weighted event count/sum per bin;
- each channel set gets its own train-only z-score standardizer;
- classifiers: multinomial Logistic Regression (`linear`) and 5-NN;
- metrics: Balanced Accuracy, Accuracy, Macro-F1.

In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate writingRing repository root')

REPO_ROOT = find_repo_root()
RESULTS_DIR = REPO_ROOT / 'notebooks/artifacts/withGyro/experiment_0_1_temporal_representation_probe/linear_angular_accel_channel_ablation_v3'
RESULTS_PATH = RESULTS_DIR / 'experiment_0_1_results.csv'
SUMMARY_PATH = RESULTS_DIR / 'experiment_0_1_summary.csv'
PAIRED_PATH = RESULTS_DIR / 'experiment_0_1_paired_gains.csv'
PAIRED_SUMMARY_PATH = RESULTS_DIR / 'experiment_0_1_paired_gain_summary.csv'
PROVENANCE_PATH = RESULTS_DIR / 'provenance.json'
for path in (RESULTS_PATH, SUMMARY_PATH, PAIRED_PATH, PAIRED_SUMMARY_PATH, PROVENANCE_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'Missing finalized artifact: {path}\nRun: bash scripts/bash_script/withGyro/submit_exp_0_1_pipeline.bash')
results = pd.read_csv(RESULTS_PATH)
summary = pd.read_csv(SUMMARY_PATH)
paired = pd.read_csv(PAIRED_PATH)
paired_summary = pd.read_csv(PAIRED_SUMMARY_PATH)
provenance = json.loads(PROVENANCE_PATH.read_text(encoding='utf-8'))
print('Results:', RESULTS_PATH)
print('Expected tasks:', provenance['expected_runs'])
print('Result rows:', len(results))

## 1. Final protocol and provenance

In [ ]:
display(pd.Series(provenance, name='value').to_frame())

## 2. Aggregate test results

In [ ]:
display(summary[[
    'channel_set', 'representation_family', 'condition', 'requested_duration_ms',
    'actual_duration_ms', 'n_bins', 'feature_dim', 'classifier', 'n_splits',
    'mean_test_balanced_accuracy', 'sd_test_balanced_accuracy',
    'mean_test_accuracy', 'mean_test_macro_f1'
]])

## 3. Fixed-duration channel-set comparison

In [ ]:
fixed = summary[summary.representation_family == 'fixed_duration'].copy()
for classifier in ('linear', '5nn'):
    fig, ax = plt.subplots(figsize=(9.2, 5.2))
    part = fixed[fixed.classifier == classifier]
    for channel_set, group in part.groupby('channel_set'):
        group = group.sort_values('actual_duration_ms')
        ax.errorbar(group.actual_duration_ms, group.mean_test_balanced_accuracy, yerr=group.sd_test_balanced_accuracy, marker='o', capsize=4, label=channel_set)
    ax.set_xlabel('Actual fixed-bin duration (ms)')
    ax.set_ylabel('Test balanced accuracy')
    ax.set_title(f'Fixed-duration channel ablation — {classifier}')
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()

## 4. Relative-progress channel-set comparison

In [ ]:
relative = summary[summary.representation_family == 'relative_progress'].copy()
for classifier in ('linear', '5nn'):
    fig, ax = plt.subplots(figsize=(8.8, 5.0))
    part = relative[relative.classifier == classifier]
    for channel_set, group in part.groupby('channel_set'):
        group = group.sort_values('n_bins')
        ax.errorbar(group.n_bins, group.mean_test_balanced_accuracy, yerr=group.sd_test_balanced_accuracy, marker='o', capsize=4, label=channel_set)
    ax.set_xlabel('Number of relative-progress bins')
    ax.set_ylabel('Test balanced accuracy')
    ax.set_title(f'Relative-progress channel ablation — {classifier}')
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()

## 5. Paired gyro contribution

In [ ]:
gain_col = 'mean_delta_balanced_accuracy_combined60_minus_accel30'
gain_sd_col = 'sd_delta_balanced_accuracy_combined60_minus_accel30'
display(paired_summary[[
    'representation_family', 'condition', 'requested_duration_ms', 'actual_duration_ms',
    'n_bins', 'classifier', 'n_splits', gain_col, gain_sd_col
]].sort_values(['classifier', 'representation_family', gain_col], ascending=[True, True, False]))

In [ ]:
fixed_gain = paired_summary[paired_summary.representation_family == 'fixed_duration'].copy()
for classifier in ('linear', '5nn'):
    group = fixed_gain[fixed_gain.classifier == classifier].sort_values('actual_duration_ms')
    fig, ax = plt.subplots(figsize=(9.0, 4.8))
    ax.errorbar(group.actual_duration_ms, group[gain_col], yerr=group[gain_sd_col], marker='o', capsize=4)
    ax.axhline(0.0, linewidth=1.0)
    ax.set_xlabel('Actual fixed-bin duration (ms)')
    ax.set_ylabel('Paired Test BA gain: combined60 - accel30')
    ax.set_title(f'Gyro-derived branch contribution — {classifier}')
    ax.grid(alpha=0.25)
    plt.show()

## 6. Best conditions per channel set

In [ ]:
best = (summary.sort_values('mean_test_balanced_accuracy', ascending=False)
        .groupby(['channel_set', 'representation_family', 'classifier'], as_index=False)
        .head(1)[['channel_set', 'representation_family', 'classifier', 'condition', 'n_bins', 'actual_duration_ms', 'feature_dim', 'mean_test_balanced_accuracy', 'sd_test_balanced_accuracy', 'mean_test_accuracy', 'mean_test_macro_f1']])
display(best)

## Interpretation

Use `accel30` vs `angular30` to compare the standalone discriminative content of the two event branches. Use `combined60 - accel30` as the primary measure of incremental information contributed by the gyro-derived angular-acceleration branch. Because `combined60` has twice the channel count of either 30-channel branch, a positive gain supports complementary information in the added branch, not greater information efficiency per channel. Linear is the main linearly-accessible-information probe; 5-NN is a secondary local-geometry probe.